# EGO Step2 F0 — Colab A100 GPU 스모크

F0 (WM-only GRPO) 학습 코드가 이 GPU 에서 **처음부터 끝까지 도는지**를 검증한다.
학습 품질/수렴은 검증 대상이 아니다 (그건 실데이터 + full run 의 몫).

런타임: **A100 40GB** 권장 (런타임 → 런타임 유형 변경 → GPU: A100).

두 모드:
- `--wm synth` (기본): 다운로드 없이 스키마-정합 합성 jsonl 로 Qwen3-VL GRPO 루프만 검증. 수 분.
- `--wm real`: 공식 V-JEPA2 ViT-g/384 백본 + EK100 AC probe 를 실제로 받아 GPU 로드·forward → 실 후보 산출 → GRPO 루프. 다운로드 ~수 GB (백본이 큼).

In [ ]:
# 1) GPU 확인
!nvidia-smi -L

In [ ]:
# 2) 리포 준비 (push 된 브랜치를 clone). 새 러너 파일이 커밋돼 있어야 한다.
%cd /content
!git clone https://github.com/hublemon/EGO.git 2>/dev/null || (cd EGO && git pull)
%cd /content/EGO

## 모드 A — 합성 데이터 스모크 (기본, 다운로드 없음)

모델 로드 → 생성 → WM-likelihood reward → LoRA 업데이트 → 체크포인트까지 검증.

In [ ]:
!python scripts/step2/colab_smoke_f0.py --wm synth

## 모드 B — 공식 V-JEPA2 실추론 스모크 (선택)

백본/probe 를 매 세션 다시 받지 않으려면 `--assets_dir` 를 Google Drive 로 지정한다.
아래 mount 셀은 선택 — Drive 캐시를 쓸 때만 실행.

In [ ]:
# (선택) Drive 마운트 — 체크포인트 캐시 재사용
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ⚠ 백본 vitg-384.pt 는 16.5GB — Drive 여유가 17GB+ 없으면 잘린 파일이 캐시된다 (무료 Drive 15GB 는 불가).
# Drive 여유가 부족하면 로컬 디스크 사용 (세션마다 재다운로드 ~수 분, Colab VM 디스크는 충분):
#   --assets_dir /content/vjepa_assets
!python scripts/step2/colab_smoke_f0.py --wm real \
    --assets_dir /content/drive/MyDrive/ego_vjepa_assets

## 모드 C — 실데이터 학습+평가 무인 파이프라인 (구 모드 C/D 통합)

단독 학습(구 모드 C)을 포괄한다: `--heldout_jsonl` 을 빼면 평가 단계를 생략하고 **학습만** 수행.
지정하면 4f-base 게이트 평가 → 500-step 검증 run(데드라인 자동 중단·125step 체크포인트)
→ trained 평가 → (잔여 시간) no_memory 평가 → session_report.md 까지 전부.

**VSCode 에서 실행하는 법**: 이 노트북을 VSCode 로 열고 → 우상단 커널 선택 → Colab → A100 런타임.
셀은 원격 Colab VM(/content)에서 실행된다.

- **Drive 마운트 주의**: `drive.mount` 의 인증 팝업은 VSCode 에서 안 뜰 수 있다.
  해결: 같은 런타임을 **브라우저 Colab 에서 한 번 열어** 마운트하면(런타임 공유) VSCode 에서도 /content/drive 가 보인다.
- **장시간 안정성**: `nohup ... &` 백그라운드 — VSCode/커널이 끊기거나 재시작돼도 계속 돈다.
  런타임 자체 회수만 예외 → `--out_root` 는 반드시 Drive (125step 체크포인트가 보험).
- **세션 재개**: `--resume_adapter <out_root>/train_val500/checkpoint-<N> --train_max_steps <잔여, 125배수> --skip_base_eval`

In [ ]:
# 10h 무인 파이프라인 — 백그라운드 실행 (한 번만 실행)
!cd /content/EGO && nohup python scripts/step2/colab_f0_10h_pipeline.py \
    --train_jsonl  /content/drive/MyDrive/ego/grpo_train.jsonl \
    --heldout_jsonl /content/drive/MyDrive/ego/grpo_heldout.jsonl \
    --path_map "/path/on/server/frames=/content/drive/MyDrive/ego/frames" \
    --out_root /content/drive/MyDrive/ego/runs/f0_v2_session1 \
    --budget_hours 10 --copy_frames_local > /content/pipeline.log 2>&1 &
print("started — 아래 모니터링 셀로 확인")

In [ ]:
# 모니터링 — 필요할 때마다 실행 (VSCode 가 끊겼다 재접속해도 동작)
!tail -n 30 /content/pipeline.log
!ls /content/drive/MyDrive/ego/runs/f0_v2_session1/train_val500/ 2>/dev/null | grep checkpoint
!tail -n 3 /content/drive/MyDrive/ego/runs/f0_v2_session1/train_val500/reward_log.jsonl 2>/dev/null

## 참고

- OOM 이면 더 작은 정책모델로: `--model Qwen/Qwen2.5-VL-7B-Instruct`
- 실데이터 jsonl 이 있으면: `--train_jsonl /content/drive/MyDrive/.../grpo_train.jsonl`
- 규모 조절 플래그: `--n_samples --max_steps --num_generations --per_device_batch --max_completion_length`
- 이미 설치된 환경이면: `--no_install`